# Experiment 1 — Device × Model × Resolution Profiling

**Objective:** Profile all YOLO26 model variants across all resolutions (640 → 320)
on all three MOT17 sequences. Collect throughput, latency, memory, and tracking-quality
metrics to produce the full device profiling table for the paper.

**Device-specific notes:**
- RPi 5 / Hailo: set `DEVICE_PROFILE=rpi5_hailo.yaml` before launching Jupyter.
- Jetson devices: export TensorRT engines first, point `model_variants` at `.engine` files.
- Development machine: desktop fallback is used automatically.

**CSV naming:** `{seq}_{model}_{imgsz}_{result_tag}.csv`  
Results from different devices land in separate files via `result_tag`.

In [ ]:
import os
from benchmark.device_profile import load_profile, apply_thread_pinning
from benchmark.config import DATA_ROOT, RESULTS_RAW, SEQUENCES, SEQ_SUFFIX, IMGSZ_BASE

PROFILE = load_profile(os.environ.get("DEVICE_PROFILE"))
# PROFILE = load_profile("rpi5_cpu.yaml")
apply_thread_pinning(PROFILE)

print(f"Device      : {PROFILE.device_label}")
print(f"Backend     : {PROFILE.backend}  |  torch device: {PROFILE.torch_device}")
print(f"Models      : {PROFILE.model_variants}")
print(f"Resolutions : {PROFILE.resolutions}")
print(f"Tag         : {PROFILE.result_tag}")

SKIP_EXISTING = False

In [ ]:
import multiprocessing as mp
import pandas as pd
from benchmark.device_profile import baked_imgsz, _BAKED_RESOLUTION_BACKENDS
from benchmark.worker import run_evaluation_isolated

ctx = mp.get_context('spawn')
failed_loads = []

for model_path in PROFILE.model_variants:
    if PROFILE.backend in ["hailo", "tensorrt_hq"] or PROFILE.backend in _BAKED_RESOLUTION_BACKENDS:
        resolutions_to_test = [baked_imgsz(model_path)]
    else:
        resolutions_to_test = PROFILE.resolutions

    for imgsz in resolutions_to_test:
        print(f"\n── Model: {model_path} ({imgsz}px) ──")

        for seq_name in SEQUENCES:
            out_csv = RESULTS_RAW / PROFILE.result_tag / "exp1" / f"{seq_name}_{model_path}_{imgsz}_{PROFILE.result_tag}.csv"

            if SKIP_EXISTING and out_csv.exists():
                print(f"  {seq_name} @ {imgsz}px: skip")
                continue

            seq_dir = DATA_ROOT / f"{seq_name}-{SEQ_SUFFIX}"
            print(f"  {seq_name} @ {imgsz}px: running ...", end=" ", flush=True)

            p = ctx.Process(
                target=run_evaluation_isolated,
                args=(PROFILE.backend, model_path, imgsz, seq_name, seq_dir, out_csv, PROFILE.torch_device),
            )
            p.start()
            p.join()

            if p.exitcode != 0:
                print(f"FAILED with exit code {p.exitcode}")
                failed_loads.append((model_path, imgsz, f"Crash with code {p.exitcode}"))
            else:
                df = pd.read_csv(out_csv)
                print(f"done — {len(df)} frames, mean {df['n_detections'].mean():.1f} det/frame")

if failed_loads:
    print("\n── Load/run failures ──")
    for name, res, reason in failed_loads:
        print(f"  {name} @ {res}px: {reason}")

In [ ]:
import numpy as np
import pandas as pd
from benchmark.device_profile import iter_model_imgsz

timing_rows = []
for model_path, imgsz in iter_model_imgsz(PROFILE):
    for seq_name in SEQUENCES:
        csv_path = RESULTS_RAW / PROFILE.result_tag / "exp1" / f"{seq_name}_{model_path}_{imgsz}_{PROFILE.result_tag}.csv"
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)

        t = df["inference_ms"].dropna()
        timing_rows.append({
            "model":       model_path,
            "imgsz":       imgsz,
            "seq":         seq_name,
            "fps":         round(1000 / t.median(), 1),
            "median_ms":   round(t.median(), 2),
            "p95_ms":      round(t.quantile(0.95), 2),
            "peak_mem_mb": round(df["mem_bytes"].max() / 1e6, 1),
            "mean_det":    round(df["n_detections"].mean(), 1),
        })

timing_df = pd.DataFrame(timing_rows)
print(timing_df.to_string(index=False))

In [ ]:
from benchmark.metrics import compute_mot_metrics

mot_rows = []
for model_path, imgsz in iter_model_imgsz(PROFILE):
    for seq_name in SEQUENCES:
        csv_path = RESULTS_RAW / PROFILE.result_tag / "exp1" / f"{seq_name}_{model_path}_{imgsz}_{PROFILE.result_tag}.csv"
        if not csv_path.exists():
            continue
        seq_dir = DATA_ROOT / f"{seq_name}-{SEQ_SUFFIX}"
        print(f"  {model_path} @ {imgsz}px / {seq_name} ...", end=" ", flush=True)
        m = compute_mot_metrics(csv_path, seq_dir)
        print(f"MOTA={m['mota']:.3f}  IDF1={m['idf1']:.3f}  IDSW/GT={m['idsw_per_gt_track']:.3f}  MT={m['mostly_tracked_ratio']:.3f}")
        mot_rows.append({"model": model_path, "imgsz": imgsz, "seq": seq_name, **m})

mot_df = pd.DataFrame(mot_rows)

In [ ]:
table1 = timing_df.merge(
    mot_df[["model", "imgsz", "seq", "mota", "idf1", "num_switches",
            "idsw_per_gt_track", "mostly_tracked_ratio", "frag_ratio"]],
    on=["model", "imgsz", "seq"],
)

for col in ["mota", "idf1", "idsw_per_gt_track", "mostly_tracked_ratio", "frag_ratio"]:
    table1[col] = table1[col].round(3)

table1 = table1.set_index(["model", "imgsz", "seq"])
print(f"\nTable I — {PROFILE.device_label} Profiling")
print(table1.to_string())

table1_path = RESULTS_RAW.parent / "profiling" / f"table1_profiling_{PROFILE.result_tag}.csv"
table1_path.parent.mkdir(parents=True, exist_ok=True)
table1.to_csv(table1_path)
print(f"\nSaved to {table1_path}")

## Results

- Key observations: *(fill after run)*
- Surprises: *(e.g., unexpected IDSW spike on dense sequence)*
- Selected model for Experiment 2: *(from cell above)*

## Next steps

- Copy `SELECTED_MODEL` value into `notebooks/02_experiment2_resolution.ipynb`.
- Repeat this notebook on each edge device; copy raw CSVs to `results/raw/` on the dev machine.
- Run `notebooks/03_results_figures.ipynb` to generate paper figures from all collected CSVs.

"Hailo-8L inference uses DFC 3.33-compiled HEF models with hardware-native mixed-precision quantisation; all other devices use PyTorch float32 .pt weights. Tracking accuracy differences between backends reflect both hardware capability and quantisation effects, which are inseparable in a real deployment scenario."